In [1]:
from pathlib import Path
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

from sklearn.dummy import DummyRegressor

import pickle


In [2]:
data_path = Path("../data/green_tripdata_2026-01.parquet")

print(f"File exists: {data_path.exists()}")
print(f"Dataset Path: {data_path}")

File exists: True
Dataset Path: ../data/green_tripdata_2026-01.parquet


In [3]:
df = pd.read_parquet(data_path)

In [4]:
df.head()

,VendorID,lpep_pickup_datetime,lpep_dropoff_datetime,store_and_fwd_flag,RatecodeID,PULocationID,DOLocationID,passenger_count,trip_distance,fare_amount,...,mta_tax,tip_amount,tolls_amount,ehail_fee,improvement_surcharge,total_amount,payment_type,trip_type,congestion_surcharge,cbd_congestion_fee
0,1,2026-01-01 00:27:58,2026-01-01 00:55:16,N,1.0,65,233,2.0,6.20,31.7,...,1.5,7.5,0.0,NaN,1.0,45.20,1.0,1.0,2.75,0.75
1,2,2026-01-01 00:44:33,2026-01-01 01:32:56,N,5.0,66,188,5.0,5.36,50.0,...,0.0,10.2,0.0,NaN,1.0,61.20,1.0,2.0,0.00,0.00
2,1,2026-01-01 00:23:45,2026-01-01 00:45:03,N,1.0,65,179,4.0,10.60,41.5,...,1.5,2.0,0.0,NaN,1.0,46.00,1.0,1.0,0.00,0.00
3,1,2026-01-01 00:44:33,2026-01-01 01:00:45,N,1.0,42,141,1.0,4.20,19.8,...,1.5,0.0,0.0,NaN,1.0,25.05,2.0,1.0,2.75,0.00
4,2,2026-01-01 00:46:04,2026-01-01 01:04:40,N,1.0,95,82,1.0,2.76,19.1,...,0.5,0.0,0.0,NaN,1.0,21.60,2.0,1.0,0.00,0.00


In [5]:
print("Rows: ", df.shape[0])
print("Columns: ", df.shape[1])
print("Shape: ", df.shape)


Rows:  40272
Columns:  21
Shape:  (40272, 21)


In [6]:
df.columns.tolist()

['VendorID',
 'lpep_pickup_datetime',
 'lpep_dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'ehail_fee',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'trip_type',
 'congestion_surcharge',
 'cbd_congestion_fee']

In [7]:
df.dtypes

VendorID                          int32
lpep_pickup_datetime     datetime64[us]
lpep_dropoff_datetime    datetime64[us]
store_and_fwd_flag               object
RatecodeID                      float64
PULocationID                      int32
DOLocationID                      int32
passenger_count                 float64
trip_distance                   float64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
ehail_fee                       float64
improvement_surcharge           float64
total_amount                    float64
payment_type                    float64
trip_type                       float64
congestion_surcharge            float64
cbd_congestion_fee              float64
dtype: object

In [8]:
important_columns = [
    "lpep_pickup_datetime",
    "lpep_dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
]

for column in important_columns:
    print(column, "->", column in df.columns)


lpep_pickup_datetime -> True
lpep_dropoff_datetime -> True
PULocationID -> True
DOLocationID -> True
trip_distance -> True


In [9]:
df[important_columns].head()

,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance
0,2026-01-01 00:27:58,2026-01-01 00:55:16,65,233,6.20
1,2026-01-01 00:44:33,2026-01-01 01:32:56,66,188,5.36
2,2026-01-01 00:23:45,2026-01-01 00:45:03,65,179,10.60
3,2026-01-01 00:44:33,2026-01-01 01:00:45,42,141,4.20
4,2026-01-01 00:46:04,2026-01-01 01:04:40,95,82,2.76


In [10]:
df[important_columns].isna().sum()

lpep_pickup_datetime     0
lpep_dropoff_datetime    0
PULocationID             0
DOLocationID             0
trip_distance            0
dtype: int64

In [11]:
df[["PULocationID", "DOLocationID", "trip_distance"]].describe()

,PULocationID,DOLocationID,trip_distance
count,40272.000000,40272.000000,40272.000000
mean,96.190480,141.878352,12.551243
std,55.683047,77.583150,1033.875580
min,1.000000,1.000000,0.000000
25%,74.000000,74.000000,1.200000
50%,75.000000,140.000000,1.960000
75%,97.000000,229.000000,3.470000
max,265.000000,265.000000,179830.920000


In [12]:
# Create the regression target: trip duration in minutes
df["duration"] = (
    df["lpep_dropoff_datetime"] - df["lpep_pickup_datetime"]
).dt.total_seconds() / 60

df[["lpep_pickup_datetime", "lpep_dropoff_datetime", "duration"]].head()

,lpep_pickup_datetime,lpep_dropoff_datetime,duration
0,2026-01-01 00:27:58,2026-01-01 00:55:16,27.300000
1,2026-01-01 00:44:33,2026-01-01 01:32:56,48.383333
2,2026-01-01 00:23:45,2026-01-01 00:45:03,21.300000
3,2026-01-01 00:44:33,2026-01-01 01:00:45,16.200000
4,2026-01-01 00:46:04,2026-01-01 01:04:40,18.600000


In [13]:
duration_stats = df["duration"].describe()
duration_stats

count    40272.000000
mean        20.075434
std         69.852105
min          0.000000
25%          8.100000
50%         12.883333
75%         20.050000
max       1439.800000
Name: duration, dtype: float64

### Observation

Most trips are relatively short, but the maximum duration is extremely
large compared with the median and upper quartile. I will inspect these
extreme records before deciding on a cleaning rule.

In [14]:
print("Negative durations:", (df["duration"] < 0).sum())
print("Zero durations:", (df["duration"] == 0).sum())

Negative durations: 0
Zero durations: 29


In [15]:
suspicious_rows = df.loc[
    df["duration"] <= 0,
    ["lpep_pickup_datetime", "lpep_dropoff_datetime", "duration"]
].rename(columns={
    "lpep_pickup_datetime": "pickup",
    "lpep_dropoff_datetime": "dropoff"
})

suspicious_rows

,pickup,dropoff,duration
511,2026-01-01 16:01:10,2026-01-01 16:01:10,0.0
2048,2026-01-03 11:03:55,2026-01-03 11:03:55,0.0
2232,2026-01-03 14:09:45,2026-01-03 14:09:45,0.0
3110,2026-01-04 14:43:04,2026-01-04 14:43:04,0.0
3875,2026-01-05 10:22:29,2026-01-05 10:22:29,0.0
4505,2026-01-05 18:54:47,2026-01-05 18:54:47,0.0
4637,2026-01-05 20:47:34,2026-01-05 20:47:34,0.0
5816,2026-01-06 20:34:57,2026-01-06 20:34:57,0.0
6199,2026-01-07 00:00:00,2026-01-07 00:00:00,0.0
7086,2026-01-07 22:35:29,2026-01-07 22:35:29,0.0


In [16]:
# Inspect long-duration trips before choosing an outlier-cleaning rule

print("Trips exceeding 60 minutes:", (df["duration"] > 60).sum())
print("Trips exceeding 120 minutes:", (df["duration"] > 120).sum())
print("Maximum duration:", df["duration"].max(), "minutes")

df.loc[
    df["duration"] > 60,
    [
        "lpep_pickup_datetime",
        "lpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "duration",
    ],
].sort_values("duration", ascending=False).head(20)

Trips exceeding 60 minutes: 957
Trips exceeding 120 minutes: 177
Maximum duration: 1439.8 minutes


,lpep_pickup_datetime,lpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,duration
19787,2026-01-18 20:20:49,2026-01-19 20:20:37,129,82,5.12,1439.800000
28925,2026-01-28 12:25:02,2026-01-29 12:24:36,129,223,2.55,1439.566667
34261,2026-01-31 14:46:50,2026-02-01 14:43:54,166,238,0.93,1437.066667
4195,2026-01-05 13:12:43,2026-01-06 13:08:16,65,132,18.06,1435.550000
23687,2026-01-22 07:20:14,2026-01-23 07:15:10,75,142,2.37,1434.933333
8442,2026-01-09 05:53:55,2026-01-10 05:45:33,129,82,7.16,1431.633333
32309,2026-01-29 20:17:11,2026-01-30 20:06:32,223,179,1.14,1429.350000
25337,2026-01-23 11:58:50,2026-01-24 11:47:48,65,132,17.86,1428.966667
33304,2026-01-30 16:55:17,2026-01-31 16:44:06,75,239,2.33,1428.816667
5619,2026-01-06 18:25:08,2026-01-07 18:13:06,42,41,0.79,1427.966667


In [17]:
df["duration"].quantile([0.01, 0.05, 0.95, 0.99])

0.01     0.066667
0.05     2.983333
0.95    44.016667
0.99    82.886500
Name: duration, dtype: float64

### Duration cleaning decision

The raw duration distribution contains zero-duration records, extremely
short trips, and a long upper tail reaching almost 24 hours.

For the baseline model, I keep trips with durations between 1 and 60
minutes. This removes clearly abnormal or unrepresentative records while
retaining the main distribution of ordinary taxi trips.

This is a baseline preprocessing rule, not a claim that every trip outside
this interval is necessarily invalid.

In [18]:
rows_before_filtering = len(df)

duration_mask = (df["duration"] >= 1) & (df["duration"] <= 60)
df = df.loc[duration_mask].copy()

rows_after_filtering = len(df)
rows_removed = rows_before_filtering - rows_after_filtering

print(f"Rows before filtering: {rows_before_filtering}")
print(f"Rows after filtering: {rows_after_filtering}")
print(f"Rows removed: {rows_removed}")

Rows before filtering: 40272
Rows after filtering: 38088
Rows removed: 2184


In [19]:
retention_percentage = (rows_after_filtering / rows_before_filtering) * 100

print(f"Retention percentage: {retention_percentage:.2f}%")

Retention percentage: 94.58%


In [20]:
df["duration"].describe()

count    38088.000000
mean        15.581492
std         10.310362
min          1.000000
25%          8.433333
50%         12.950000
75%         19.633333
max         60.000000
Name: duration, dtype: float64

In [21]:
print(f"Minimum duration: {df['duration'].min():.2f} minutes")
print(f"Maximum duration: {df['duration'].max():.2f} minutes")

assert df["duration"].min() >= 1
assert df["duration"].max() <= 60

print("Proof passed: all durations are between 1 and 60 minutes.")
print(f"Durations < 1 minute: {(df['duration'] < 1).sum()}")
print(f"Durations > 60 minutes: {(df['duration'] > 60).sum()}")

Minimum duration: 1.00 minutes
Maximum duration: 60.00 minutes
Proof passed: all durations are between 1 and 60 minutes.
Durations < 1 minute: 0
Durations > 60 minutes: 0


In [22]:
distance_stats = df["trip_distance"].quantile([0.01, 0.05, 0.95, 0.99])

print("Trip-distance summary:")
print(distance_stats)

print("\nTrip-distance counts:")
print("Distance == 0:", (df["trip_distance"] == 0).sum())
print("Distance < 0:", (df["trip_distance"] < 0).sum())
print("Distance > 50:", (df["trip_distance"] > 50).sum())
print("Distance > 100:", (df["trip_distance"] > 100).sum())

largest_distance_trips = (
    df.loc[
        df["trip_distance"].nlargest(20).index,
        [
            "trip_distance",
            "duration",
            "PULocationID",
            "DOLocationID",
        ],
    ]
    .sort_values("trip_distance", ascending=False)
)

print("\nTrips with the largest trip distances:")
largest_distance_trips

Trip-distance summary:
0.01     0.0000
0.05     0.5600
0.95     8.9600
0.99    15.5626
Name: trip_distance, dtype: float64

Trip-distance counts:
Distance == 0: 549
Distance < 0: 0
Distance > 50: 6
Distance > 100: 6

Trips with the largest trip distances:


,trip_distance,duration,PULocationID,DOLocationID
38985,179830.92,11.000000,244,166
39635,82698.45,15.000000,42,127
38509,39298.38,5.000000,223,223
38960,33216.30,35.000000,61,132
39062,32319.24,26.000000,61,264
35538,13439.72,16.000000,260,7
2831,37.39,49.566667,195,265
37327,36.03,41.000000,244,44
149,35.50,49.400000,130,265
7087,30.20,45.816667,97,265


In [23]:
df["trip_distance"].describe()

count     38088.000000
mean         12.943096
std        1063.101308
min           0.000000
25%           1.240000
50%           1.970000
75%           3.380000
max      179830.920000
Name: trip_distance, dtype: float64

### Trip-distance cleaning decision

Most trip distances are small: the 99th percentile is about 15.56 miles.
The dataset contains six extreme distance values above 100 miles, including
values in the tens of thousands of miles, despite trip durations of only a
few minutes.

For the baseline model, I keep records with trip distances greater than
0 and no more than 50 miles. This removes zero-distance records and the
clearly corrupted extreme values while retaining the normal distance
distribution.

In [24]:
rows_before_distance_cleaning = len(df)

distance_mask = (df["trip_distance"] > 0) & (df["trip_distance"] <= 50)
df = df.loc[distance_mask].copy()

rows_after_distance_cleaning = len(df)
rows_removed_distance = (
    rows_before_distance_cleaning - rows_after_distance_cleaning
)
distance_retention_percentage = (
    rows_after_distance_cleaning / rows_before_distance_cleaning * 100
)

print(f"Rows before distance cleaning: {rows_before_distance_cleaning}")
print(f"Rows after distance cleaning: {rows_after_distance_cleaning}")
print(f"Rows removed: {rows_removed_distance}")
print(f"Retention percentage: {distance_retention_percentage:.2f}%")

print(f"\nMinimum trip_distance: {df['trip_distance'].min():.2f}")
print(f"Maximum trip_distance: {df['trip_distance'].max():.2f}")
print(f"Number <= 0: {(df['trip_distance'] <= 0).sum()}")
print(f"Number > 50: {(df['trip_distance'] > 50).sum()}")

assert df["trip_distance"].min() > 0
assert df["trip_distance"].max() <= 50
assert (df["trip_distance"] <= 0).sum() == 0
assert (df["trip_distance"] > 50).sum() == 0

print("\nProof passed: all trip distances are greater than 0 and no more than 50 miles.")

Rows before distance cleaning: 38088
Rows after distance cleaning: 37533
Rows removed: 555
Retention percentage: 98.54%

Minimum trip_distance: 0.01
Maximum trip_distance: 37.39
Number <= 0: 0
Number > 50: 0

Proof passed: all trip distances are greater than 0 and no more than 50 miles.


In [25]:
# Create the categorical pickup/dropoff location feature
df["PU_DO"] = (
    df["PULocationID"].astype(str)
    + "_"
    + df["DOLocationID"].astype(str)
)

# Inspect the first five rows
df[["PULocationID", "DOLocationID", "PU_DO"]].head()

# Calculate unique values
print("Unique PULocationID values:", df["PULocationID"].nunique())
print("Unique DOLocationID values:", df["DOLocationID"].nunique())
print("Unique PU_DO values:", df["PU_DO"].nunique())

Unique PULocationID values: 228
Unique DOLocationID values: 240
Unique PU_DO values: 5129


In [26]:
# Define the model inputs and target predictions
categorical = ["PU_DO"]
numerical = ["trip_distance"]
target = "duration"

features = categorical + numerical

print("Features:", features)
print("Target:", target)

Features: ['PU_DO', 'trip_distance']
Target: duration


In [27]:
# Split the dataset into training and validation sets
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

total_rows = len(df)
train_rows = len(train_df)
val_rows = len(val_df)

print("Total cleaned rows", total_rows)
print("Training rows", train_rows)
print("Validation rows", val_rows)

print(f"Training set percentage: {train_rows / total_rows * 100:.2f}%")
print(f"Validation set percentage: {val_rows / total_rows * 100:.2f}%")

print("Row-count check:", train_rows + val_rows == total_rows)

Total cleaned rows 37533
Training rows 30026
Validation rows 7507
Training set percentage: 80.00%
Validation set percentage: 20.00%
Row-count check: True


In [28]:
# Separate the target from the training and validation sets
y_train = train_df[target]
y_val = val_df[target]

print("Training target type:", type(y_train))
print("Training target shape:", y_train.shape)
print("Validation target shape:", y_val.shape)

Training target type: <class 'pandas.core.series.Series'>
Training target shape: (30026,)
Validation target shape: (7507,)


In [29]:
# Convert each row of selected features into a dictionary
train_dicts = train_df[features].to_dict(orient="records")
val_dicts = val_df[features].to_dict(orient="records")

train_dicts[:2]

[{'PU_DO': '232_127', 'trip_distance': 13.79},
 {'PU_DO': '74_41', 'trip_distance': 0.4}]

In [30]:
# Learn the feature vocab from the training set
dv = DictVectorizer(sparse=True)

X_train = dv.fit_transform(train_dicts)

X_val = dv.transform(val_dicts)

In [31]:
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)

print("y_train shape:", y_train.shape)
print("y_val shape:", y_val.shape)

X_train shape: (30026, 4494)
X_val shape: (7507, 4494)
y_train shape: (30026,)
y_val shape: (7507,)


In [32]:
print(
    "Training rows match:",
    X_train.shape[0] == y_train.shape[0],
)

print(
    "Validation rows match:",
    X_val.shape[0] == y_val.shape[0],
)

Training rows match: True
Validation rows match: True


In [33]:
feature_names = dv.get_feature_names_out()

print("Number of encoded features:", len(feature_names))

print("\nFirst 10 feature names:")
print(feature_names[:10])

print("\nLast 10 feature names:")
print(feature_names[-10:])


Number of encoded features: 4494

First 10 feature names:
['PU_DO=100_116' 'PU_DO=100_247' 'PU_DO=100_35' 'PU_DO=100_74'
 'PU_DO=100_86' 'PU_DO=101_95' 'PU_DO=101_98' 'PU_DO=102_177'
 'PU_DO=102_37' 'PU_DO=102_82']

Last 10 feature names:
['PU_DO=97_94' 'PU_DO=97_97' 'PU_DO=98_10' 'PU_DO=98_121' 'PU_DO=98_132'
 'PU_DO=98_216' 'PU_DO=98_218' 'PU_DO=98_95' 'PU_DO=98_98' 'trip_distance']


In [34]:
assert len(train_df) + len(val_df) == len(df)

assert X_train.shape[0] == len(y_train)
assert X_val.shape[0] == len(y_val)

assert X_train.shape[1] == X_val.shape[1]

print("All assertions passed: training and validation sets are consistent.")

All assertions passed: training and validation sets are consistent.


## Train the baseline model

Train a linear regression model using the encoded `PU_DO` and
`trip_distance` features, then evaluate it on the validation set.

In [35]:
model = LinearRegression()

model.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [36]:
y_pred = model.predict(X_val)


In [37]:
print("Predictions shape:", y_pred.shape)
print("Targets shape:", y_val.shape)

Predictions shape: (7507,)
Targets shape: (7507,)


In [38]:
prediction_comparison = pd.DataFrame({
    "actual_duration": y_val.to_numpy(),
    "predicted_duration": y_pred,
})

prediction_comparison.head(10)

,actual_duration,predicted_duration
0,21.200000,11.798419
1,4.166667,5.890020
2,3.583333,6.782106
3,11.800000,12.719802
4,13.450000,20.164787
5,23.633333,22.123797
6,15.966667,15.560057
7,20.766667,24.102776
8,8.783333,12.134118
9,14.700000,16.292247


In [39]:
mae = mean_absolute_error(y_val, y_pred)

print(f"Mean Absolute Error (MAE): {mae:.2f} minutes")

Mean Absolute Error (MAE): 4.22 minutes


In [40]:
rmse = mean_squared_error(y_val, y_pred) ** 0.5
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} minutes")

Root Mean Squared Error (RMSE): 6.51 minutes


In [41]:
assert len(y_pred) == len(y_val)
assert mae >= 0
assert rmse >= 0

print("Model evaluation checks passed.")

Model evaluation checks passed.


In [42]:
dummy_model = DummyRegressor(strategy="median")

dummy_model.fit(X_train, y_train)

dummy_pred = dummy_model.predict(X_val)

In [43]:
dummy_mae = mean_absolute_error(y_val, dummy_pred)
dummy_rmse = mean_squared_error(y_val, dummy_pred) ** 0.5

print(f"Dummy MAE:  {dummy_mae:.2f} minutes")
print(f"Dummy RMSE: {dummy_rmse:.2f} minutes")

print(f"Linear Regression MAE:  {mae:.2f} minutes")
print(f"Linear Regression RMSE: {rmse:.2f} minutes")

Dummy MAE:  7.30 minutes
Dummy RMSE: 10.56 minutes
Linear Regression MAE:  4.22 minutes
Linear Regression RMSE: 6.51 minutes


In [44]:
print("Minimum prediction:", y_pred.min())
print("Maximum prediction:", y_pred.max())

print("Predictions <= 0:", (y_pred <= 0).sum())
print("Predictions > 60:", (y_pred > 60).sum())

Minimum prediction: 0.8079409153467845
Maximum prediction: 90.58497004729368
Predictions <= 0: 0
Predictions > 60: 24


In [45]:
y_train_pred = model.predict(X_train)

train_mae = mean_absolute_error(y_train, y_train_pred)
train_rmse = mean_squared_error(y_train, y_train_pred) ** 0.5

print(f"Training MAE:   {train_mae:.2f}")
print(f"Validation MAE: {mae:.2f}")

print(f"Training RMSE:   {train_rmse:.2f}")
print(f"Validation RMSE: {rmse:.2f}")

Training MAE:   3.03
Validation MAE: 4.22
Training RMSE:   4.65
Validation RMSE: 6.51


## Save the baseline model

Save the fitted `DictVectorizer` and `LinearRegression` model together so
the same preprocessing vocabulary can be reused during prediction.

In [46]:
model_path = Path("../models/baseline.pkl")

model_path.parent.mkdir(parents=True, exist_ok=True)

In [47]:
baseline_artifact = {
    "vectorizer": dv,
    "model": model,
}

with model_path.open("wb") as f:
    pickle.dump(baseline_artifact, f)

print("Model saved to:", model_path)
print("File exists:", model_path.exists())
print("File size:", model_path.stat().st_size, "bytes")

Model saved to: ../models/baseline.pkl
File exists: True
File size: 139418 bytes
